# The holding-based attribution: allocation, currency and the link

*A learning exercise performed in role: a simulated mandate with no client and no institution. Nothing in this notebook is investment advice, a recommendation, or a client communication.*

**Entry point** `python3 -m portfolio_workbench.attribute.brinson`

**Modules covered** `attribute/brinson.py`

_Generated from the code by `python3 -m reporting.notebooks`: the module headers below are read out of the modules themselves, and the run is the entry point's own output._

## 1. What this module does, and the source of every method in it

The attribution layer explains each cell's active return from its holdings. The holding-based view splits the active return into allocation, a currency line and their cross product, links the monthly effects so that they sum to the compounded excess, and names the cost line. The entry point prints each cell's lines, the link's residual and the cost the grid actually charged.

**Sources.** Every public function of the modules this notebook covers, and what it traces to. The map is checked over the code by the acceptance fixture, so a method added without a source fails a command rather than going unnoticed.

- `attribute/brinson.py::allocation_only` traces to the holding-based decomposition: Brinson & Fachler (1985), Journal of Portfolio Management 11(3), and Brinson, Hood & Beebower (1986), Financial Analysts Journal 42(4), reported allocation-only because one instrument per sleeve leaves no selection term to compute
- `attribute/brinson.py::carino` traces to the reported link: Carino (1999), Journal of Performance Measurement 3(4)
- `attribute/brinson.py::cost_line` traces to the cost convention this effort pinned by execution: cost is charged on traded notional, `2 x one-way turnover x per-side rate`, so a factor of two cannot return silently
- `attribute/brinson.py::decompose` traces to the holding-based decomposition: Brinson & Fachler (1985), Journal of Portfolio Management 11(3), and Brinson, Hood & Beebower (1986), Financial Analysts Journal 42(4), reported allocation-only because one instrument per sleeve leaves no selection term to compute
- `attribute/brinson.py::linking` traces to the linking comparison itself: Cariño's smoothing factor against Menchero's single constant plus a period adjustment, with both landed on the compounded excess
- `attribute/brinson.py::main` traces to the holding-based decomposition: Brinson & Fachler (1985), Journal of Portfolio Management 11(3), and Brinson, Hood & Beebower (1986), Financial Analysts Journal 42(4), reported allocation-only because one instrument per sleeve leaves no selection term to compute
- `attribute/brinson.py::menchero` traces to the cross-check link: Menchero (2000), Journal of Performance Measurement, Fall, 36-42
- `attribute/brinson.py::report` traces to the holding-based decomposition: Brinson & Fachler (1985), Journal of Portfolio Management 11(3), and Brinson, Hood & Beebower (1986), Financial Analysts Journal 42(4), reported allocation-only because one instrument per sleeve leaves no selection term to compute
- `attribute/brinson.py::single_period` traces to the single-period arithmetic of Brinson & Fachler (1985) and Brinson, Hood & Beebower (1986): a weight deviation against the benchmark's own return

## 2. Why it works this way, including what was rejected

_The module headers, verbatim: each records why the module is shaped the way it is, what was rejected, and the measurement that settled it. They are quoted here rather than restated, so the notebook cannot drift from the code._

**`attribute/brinson.py`**

The holding-based decomposition: Brinson-Fachler, the currency dimension, and linking.

**The decomposition is allocation-only, structurally, and it says so rather than printing zeros.**
Brinson-Fachler (1985, JPM 11(3), 73-76) splits the active return into allocation, selection and
interaction. Its selection term is the benchmark's weight applied to the difference between the
portfolio's sleeve return and the benchmark's, and in this universe those are the same number: the
policy benchmark holds the *same instrument* for each sleeve, so `r_p,i = r_b,i` and selection is
identically zero. The workbook therefore states that the term does not exist here, with the reason,
rather than carrying a zero column - a column of zeros in a performance table reads as a market
finding, and this one is a property of how the benchmark was built.

**The currency dimension carries the interaction instead.** The euro return of a sleeve is
`(1 + local)(1 + translation) - 1`, so the active return decomposes along exactly that line into
allocation measured on **local-currency** returns, a currency effect that is the active weight applied
to the translation, and the cross product of the two, which is named as its own line rather than
folded into either. The lines then sum to the active return exactly, and measuring allocation on euro
returns while also adding a currency line would count the translation twice - which is why the
ordering is stated here and not left to the reader. Two limits travel with it: the dimension is
**quotation-currency, not look-through**, since an unhedged USD holding inside a euro-quoted ETF moves
the fund's NAV with nothing in this data to separate it; and only the SEK-quoted sleeve shows a
translation at all.

**"Reconciles exactly" is a linking property once more than one period is joined.** The single-period
identity is arithmetic; the compounded one is not, because the effects do not compound with the
returns that produced them. Cariño (1999) is the reported method and Menchero (2000) the cross-check:
both are implemented, both are held against the compounded geometric excess, and the difference
between them is reported rather than resolved away. Cariño's coefficient is one logarithm per period
and is exact by construction; Menchero's is a constant for the run plus a period adjustment carrying
the compounding residual in proportion to each period's own active return. They land on the same total
and distribute it differently, which is exactly why the second is worth printing.

**Costs are a named line and the benchmark is costless by convention.** A policy benchmark is not
billed for its own rebalancing, so gross reconciles to it; the portfolio is charged the per-side rate
on traded notional and net equals gross minus cost. The cost is never merged into selection, where it
would read as a stock-picking result.

## 3. The data contract it consumes, and the as-of rule

One weight per sleeve against the policy benchmark's weights, on the panel's own currency split. A sleeve's euro return is `(1 + local)(1 + translation) - 1`, so allocation is measured on **local-currency** returns and the currency move is its own line: measuring allocation in euro and then adding a currency line counts the translation twice, and the double count is invisible because the lines still sum. Selection and interaction do not exist here and are stated as structural rather than printed as zeros: the policy benchmark holds the same instrument for each sleeve, so `r_p,i = r_b,i` for every sleeve.

## 4. The worked example on small numbers, with the identity checked

The Brinson arithmetic on two sleeves and one month, with the three lines summing to the active return by construction, and the level the link is measured against taken in the frame the rest of the package reports in.

The cell below runs on numbers small enough to check by hand and asserts the identity, so a reader can see the arithmetic rather than take the module's word for it.

In [1]:
import numpy as np

from portfolio_workbench.attribute import brinson

portfolio_weights = np.array([0.60, 0.40])
benchmark_weights = np.array([0.50, 0.50])
portfolio_returns = np.array([0.02, 0.01])
benchmark_returns = np.array([0.01, 0.02])

single = brinson.single_period(portfolio_weights, benchmark_weights, portfolio_returns, benchmark_returns)
assert abs(single["totals"]["allocation"] + single["totals"]["interaction"] - single["active"]) < 1e-15
# Allocation uses the benchmark's own return as the reference level, which is what makes the sum
# identify the active return rather than a mixture of two framings.
assert abs(single["totals"]["allocation"] - (0.10 * (0.01 - 0.015) + -0.10 * (0.02 - 0.015))) < 1e-15
assert single["source"] == brinson.BRINSON_FACHLER
print("allocation", round(single["totals"]["allocation"], 6), "interaction", round(single["totals"]["interaction"], 6),
      "active", round(single["active"], 6))

allocation -0.001 interaction 0.002 active 0.001


## 5. The real run: inputs, parameters, provenance block

The provenance block is printed first, then the parameters this module decides under, then the entry point's own report. The report is the module's output rather than a transcription of it, so a number quoted from a notebook is the number the module prints.

In [2]:
from portfolio_workbench.data import loader, universe

document = loader.load_panel()
months = document.months
print(f"snapshot {document.snapshot_id}, taken as of {document.as_of}")
print(f"panel {len(months)} months {months.min()}..{months.max()} across {len(universe.TICKERS)} sleeves")
print("manifest fields: " + ", ".join(sorted(document.manifest)))

from portfolio_workbench.attribute import brinson

print(f"reported form: {brinson.BRINSON_FACHLER}")
print(f"cross-check form: {brinson.BRINSON_HOOD_BEEBOWER}")
print(f"link: {brinson.CARINO}")

snapshot 2026-09-13, taken as of 2026-09-13T07:56:28+00:00
panel 191 months 2010-09..2026-07 across 11 sleeves
manifest fields: created, excluded, files, instruments, snapshot_id, window
reported form: Brinson & Fachler (1985), JPM 11(3), 73-76
cross-check form: Brinson, Hood & Beebower (1986), FAJ 42(4), 39-44
link: Carino (1999), Journal of Performance Measurement 3(4)


In [3]:
import subprocess
import sys

finished = subprocess.run(
    [sys.executable, "-m", "portfolio_workbench.attribute.brinson"], capture_output=True, text=True, cwd="."
)
print(finished.stdout)
assert finished.returncode == 0, finished.stderr

[attrib] snapshot 2026-09-13: 20 runs, 2015-09..2026-07 (131 months)
[attrib] Brinson & Fachler (1985), JPM 11(3), 73-76, arithmetic; Carino (1999), Journal of Performance Measurement 3(4) reported, Menchero (2000), Journal of Performance Measurement, Fall, 36-42 as the cross-check
[attrib] allocation is measured on LOCAL-currency returns, so selection here is absent by structure - the selection term does not exist here: the policy benchmark holds the same instrument for each sleeve, so the portfolio's sleeve return and the benchmark's are the same number and the term is identically zero. Stated rather than printed as a zero column, which would read as a finding
[attrib] cost: 10 bp per side on traded notional, which is twice the one-way turnover, charged to the net series; the benchmark is costless by convention, and the cost is never merged into selection
[attrib] cell                               allocation  currency interaction gross excess   residual  carino-menchero
[attrib] equ

## 6. Results, and how to read them, including the resolution limit and what a reader must not conclude

The linked effects sum to the compounded geometric excess, and the residual is reported rather than absorbed, so a reader can see how much of the answer the link carries. Allocation-only is a statement about this universe, not about the method: with one instrument per sleeve there is no security selection to measure. The cost line is the charge the grid applied, so a difference between cells after cost is a difference the account has already paid for. A reader must not read the currency line as a decision a manager took: it is the translation the sleeve was exposed to.

## 7. What this module does not establish

Nothing here establishes that allocation was a deliberate exposure rather than a drift the benchmark rule produced. It does not measure security selection, because the universe has none to measure, and it says nothing about the sleeves' internal holdings.